# Start of session
Run this first in every new Colab session (runtime type: T4 GPU). See D-007, D-017 and C-002 to C-006.

In [ ]:
# ===== START OF SESSION: run this first, every new Colab session =====
import os, re, sys, subprocess, shutil, time
from pathlib import Path
from importlib.metadata import version
from google.colab import drive, userdata

USER, REPO_NAME = "yarushahemmanuel10", "vehicle-damage-vision"
REPO       = Path("/content") / REPO_NAME
ZIP_DRIVE  = Path("/content/drive/MyDrive/vdv/data/CarDD_release.zip")
DATA_LOCAL = Path("/content/data")                 # fast local disk, outside the repo
TESTED     = {"torch": "2.11.0+cu130", "torchvision": "0.26.0+cu130"}

# 1) Mount Google Drive
drive.mount("/content/drive")

# 2) Token-safe git: the token only ever exists in one git command's environment
ASKPASS = Path("/content/git_askpass.sh")          # outside the repo; contains no secret
ASKPASS.write_text('#!/bin/sh\ncase "$1" in\n  Username*) echo "x-access-token" ;;\n  *) echo "$GIT_TOKEN" ;;\nesac\n')
ASKPASS.chmod(0o700)

def git(*args, auth=False, check=False):
    """Run git (inside the repo, except clone). auth=True supplies the token via askpass.
    check=True raises an error if git fails."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if auth:
        env.update(GIT_ASKPASS=str(ASKPASS), GIT_TOKEN=userdata.get("GITHUB_TOKEN"))
    cmd = ["git", *args] if args[0] == "clone" else ["git", "-C", str(REPO), *args]
    r = subprocess.run(cmd, capture_output=True, text=True, env=env)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed (exit code {r.returncode}): stopping here")
    return r.returncode

def git_push():
    return git("push", "origin", "main", auth=True)

# 3) Latest code: clone in a new session, pull if already here. Fails loudly.  [C-004]
if REPO.exists():
    git("pull", "--ff-only", auth=True, check=True)
else:
    git("clone", f"https://github.com/{USER}/{REPO_NAME}.git", str(REPO), auth=True, check=True)
git("config", "user.name", "Yarushah Emmanuel Sardar")
git("config", "user.email", "129624872+yarushahemmanuel10@users.noreply.github.com")
os.chdir(REPO)

# 4) Check PyTorch is the tested build; never reinstall it.  [C-002]
for pkg, want in TESTED.items():
    got = version(pkg)
    if got != want:
        raise RuntimeError(f"{pkg} is {got}, but this project was tested on {want}. "
                           "Colab's image has changed: stop and decide before training.")
Path("/content/constraints.txt").write_text("".join(f"{p}=={v}\n" for p, v in TESTED.items()))

# 5) Session requirements = requirements.txt minus torch/torchvision, written to a file  [C-005]
session_reqs = []
for line in Path("requirements.txt").read_text().splitlines():
    s = line.strip()                                       # filter on the stripped line
    if not s or s.startswith("#"):
        continue
    name = re.split(r"[=<>!~;\[\s]", s, maxsplit=1)[0].lower()
    if name in {"torch", "torchvision"}:
        continue
    session_reqs.append(s)
Path("/content/reqs_session.txt").write_text("\n".join(session_reqs) + "\n")
print(f"Session requirements: {len(session_reqs)} packages (torch/torchvision excluded)")

%pip install -q -r /content/reqs_session.txt -e . -c /content/constraints.txt
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))          # lets THIS kernel import vdv without a restart

# 6) Data: copy the zip to local disk and unzip, once per session
marker = DATA_LOCAL / ".unzipped"
if marker.exists():
    print("Data already on local disk.")
elif ZIP_DRIVE.exists():
    t0 = time.time()
    DATA_LOCAL.mkdir(parents=True, exist_ok=True)
    local_zip = Path("/content/CarDD_release.zip")
    shutil.copy(ZIP_DRIVE, local_zip)                              # one big sequential read
    subprocess.run(["unzip", "-o", "-q", str(local_zip), "-d", str(DATA_LOCAL)], check=True)  # [C-003]
    local_zip.unlink()                                             # deletes the TEMP copy only
    marker.touch()                                                 # only after a successful unzip
    print(f"Data ready in {time.time() - t0:.0f}s:", sorted(p.name for p in DATA_LOCAL.iterdir()))
else:
    print("WARNING: zip not found on Drive at", ZIP_DRIVE)

# 7) Classification crops: restore the version the CURRENT code expects, refuse any other  [D-017]
from vdv.data.crops import crop_code_version, check_crop_cache
CROP_VERSION_NOW = crop_code_version()
CROP_DIR  = DATA_LOCAL / f"crops_{CROP_VERSION_NOW}"
CROP_ZIP  = ZIP_DRIVE.parent / f"crops_{CROP_VERSION_NOW}.zip"
try:
    check_crop_cache(CROP_DIR)
    print(f"Crops {CROP_VERSION_NOW} already on local disk.")
except (FileNotFoundError, RuntimeError):
    if CROP_ZIP.exists():
        t0 = time.time()
        local_crop_zip = Path("/content") / CROP_ZIP.name
        shutil.copy(CROP_ZIP, local_crop_zip)
        subprocess.run(["unzip", "-o", "-q", str(local_crop_zip), "-d", str(DATA_LOCAL)], check=True)
        local_crop_zip.unlink()                                    # deletes the TEMP copy only
        meta = check_crop_cache(CROP_DIR)                          # raises if version or PNG count is wrong
        print(f"Crops {CROP_VERSION_NOW} restored in {time.time() - t0:.0f}s: {meta['n_crops']} crops")
    else:
        found = sorted(p.name for p in ZIP_DRIVE.parent.glob("crops_*.zip"))
        print(f"WARNING: no crop zip for version {CROP_VERSION_NOW} on Drive (found: {found}). "
              "Rebuild with scripts/build_crops.py before training.")

# 8) Checks: package, GPU, current commit
import vdv, torch
print("vdv from:", vdv.__file__)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU: change runtime type")
git("log", "--oneline", "-1")